In [ ]:
# ⚠️ RUN ALL CELLS FROM TOP AFTER KERNEL RESTART


from pathlib import Path

#  CHANGE THIS PATH AS PER YOUR SYSTEM
BASE_PATH = Path(r"C:\Users\Asus\OneDrive\Desktop\estore_project")

# Folder structure
folders = [
    BASE_PATH / "data" / "raw",
    BASE_PATH / "data" / "processed",
    BASE_PATH / "etl",
    BASE_PATH / "app",
    BASE_PATH / "powerbi"
]

# Files to create
files = [
    BASE_PATH / "data" / "raw" / "report.csv",
    BASE_PATH / "data" / "processed" / "clean.csv",
    BASE_PATH / "etl" / "cleaning.py",
    BASE_PATH / "etl" / "transform.py",
    BASE_PATH / "app" / "input_app.py",
    BASE_PATH / "powerbi" / "dashboard.pbix",
    BASE_PATH / "requirements.txt"
]

# Create folders
for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

# Create empty files
for file in files:
    file.touch(exist_ok=True)

print("✅ E-commerce automation project structure created successfully!")


In [ ]:
import pandas as pd

df = pd.read_csv("C:/Users/Asus/OneDrive/Desktop/estore_project/data/raw/report.csv")


In [ ]:
df.head()

In [ ]:
import pandas as pd

# -----------------------------
# 1. Load raw data
# -----------------------------
raw_path = "C:/Users/Asus/OneDrive/Desktop/estore_project/data/raw/report.csv"
df = pd.read_csv(raw_path)

# -----------------------------
# 2. Remove unwanted index column if exists
# -----------------------------
if df.columns[0].lower().startswith("unnamed"):
    df.drop(columns=df.columns[0], inplace=True)

# -----------------------------
# 3. Standardize column names
# -----------------------------
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace(".", "", regex=False)
)

# -----------------------------
# 4. Convert numeric columns
# -----------------------------
numeric_cols = [
    "payable_amount",
    "amount_paid_to_company",
    "total_remaining_amount"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# -----------------------------
# 5. Handle phone numbers
# -----------------------------
phone_cols = [
    "distributor_phone_number",
    "vle_phone_number",
    "alternate_phone_number"
]

for col in phone_cols:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace(".0", "", regex=False)
            .replace("nan", "")
        )

# -----------------------------
# 6. Handle GST columns
# -----------------------------
gst_cols = [
    "company_gst_no",
    "distributor_gst_no",
    "vle_gst_no"
]

for col in gst_cols:
    if col in df.columns:
        df[col] = df[col].fillna("NOT_AVAILABLE")

# -----------------------------
# 7. Convert date columns
# -----------------------------
date_cols = ["lead_bill_date"]

for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

# -----------------------------
# 8. Business metrics
# -----------------------------
df["order_year"] = df["lead_bill_date"].dt.year
df["order_month"] = df["lead_bill_date"].dt.month

# -----------------------------
# 9. Save cleaned data
# -----------------------------
output_path = "C:/Users/Asus/OneDrive/Desktop/estore_project/data/processed/clean.csv"
df.to_csv(output_path, index=False)

print("✅ Data cleaned & saved for Power BI")


In [ ]:
df = pd.read_csv(output_path)

In [ ]:
df.head()

In [ ]:
keep_cols = [
    "state",
    "district",
    "company",
    "csc_id",
    "distributor_name",
    "order_number",
    "order_date",
    "payable_amount",
    "amount_paid_to_company",
    "total_remaining_amount",
]

df = df[[c for c in keep_cols if c in df.columns]]


In [ ]:
df.head()

In [ ]:
# -----------------------------
# Drop unwanted columns
# -----------------------------
drop_cols = [
    "sr_no",
    "alternate_name",
    "alternate_phone_number",
    "company_gst_no",
    "distributor_gst_no",
    "vle_gst_no"
]

df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)


In [ ]:
# Function to check if a string contains non-ASCII characters
def has_non_ascii(s):
    try:
        return any(ord(c) > 127 for c in str(s))
    except:
        return False

# Apply to all string/object columns
string_cols = df.select_dtypes(include='object').columns
mask = pd.DataFrame(False, index=df.index, columns=string_cols)

for col in string_cols:
    mask[col] = df[col].apply(has_non_ascii)

# Rows containing any non-ASCII
rows_with_garbage = mask.any(axis=1)
print(f"Found {rows_with_garbage.sum()} rows with non-ASCII characters.")

# Preview unwanted rows
print(df[rows_with_garbage])


In [ ]:
output_path = "C:/Users/Asus/OneDrive/Desktop/estore_project/data/processed/clean.csv"
df.to_csv(output_path, index=False)

print("✅ Data cleaned & saved for Power BI")

In [ ]:
df = pd.read_csv(output_path)

In [ ]:
df.head()

In [ ]:
vle_df = pd.read_csv("C:/Users/Asus/OneDrive/Desktop/estore_project/data/raw/estorevle.csv")

vle_df.head()

In [ ]:
if vle_df.columns[0].lower().startswith("unnamed"):
    vle_df.drop(columns=df.columns[0], inplace=True)

# -----------------------------
# 3. Standardize column names
# -----------------------------
vle_df.columns = (
    vle_df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace(".", "", regex=False)
)

vle_df.rename(columns={'csc_code': 'csc_id'}, inplace=True)

# Save cleaned data
# -----------------------------
output_path = "C:/Users/Asus/OneDrive/Desktop/estore_project/data/processed/vle.csv"
vle_df.to_csv(output_path, index=False)

print("✅ VLE ID Data cleaned")



In [ ]:
vle = pd.read_csv(r"C:/Users/Asus/OneDrive/Desktop/estore_project/data/processed/vle.csv")

vle.head()

In [ ]:

# Count of nulls
null_count = vle['csc_id'].isnull().sum()
print(f"Number of null values in CSC ID: {null_count}")

# Optional: show the rows with null CSC ID
null_rows = vle[vle['csc_id'].isnull()]
print(null_rows)


In [ ]:
print(df['csc_id'].dtype)
print(df['csc_id'].head(10))
print(df['csc_id'].unique()[:10])


In [ ]:
df['csc_id'] = (
    df['csc_id']
    .astype(str)
    .str.strip()
    .str.replace('.0', '', regex=False)
)

# Remove invalid values
df = df[df['csc_id'].notna()]
df = df[df['csc_id'] != 'nan']


In [ ]:
def order_summary(df, csc_id):
    orders = df[df['csc_id'] == str(csc_id)]

    return {
        "total_orders": orders.shape[0],
        "total_order_value": orders['payable_amount'].sum(),
        "total_paid": orders['amount_paid_to_company'].sum(),
        "total_remaining": orders['total_remaining_amount'].sum(),
    }


In [ ]:
summary = order_summary(df, "565752450012")
summary

In [ ]:
df['csc_id_key'] = df['csc_id'].astype(str).str.strip()


In [ ]:
df['csc_id_key'].isna().sum()


In [ ]:
input_csc = input("Enter CSC ID: ")
input_csc = str(input_csc).strip()
df[df['csc_id_key'] == input_csc]


In [2]:
import sys
import os

# Add the specific folder to Python path
project_path = r"C:/Users/Asus/OneDrive/Desktop/estore_project"  # <-- full path to the folder containing ecommerce_pipeline.py
if project_path not in sys.path:
    sys.path.append(project_path)

# Now import your module
import ecommerce_pipeline

# Test
print(ecommerce_pipeline.__file__)


C:\Users\Asus\OneDrive\Desktop\estore_project\ecommerce_pipeline.py


In [2]:
import importlib
import ecommerce_pipeline

importlib.reload(ecommerce_pipeline)
orders_df, vleid_df = ecommerce_pipeline.run_pipeline()


📥 Processing Orders data...
✅ Orders cleaned & saved
📥 Processing VLE ID data...
✅ VLE ID cleaned & saved

🎉 Ecommerce ETL pipeline executed successfully!
Orders DataFrame: 3427 rows
VLE ID DataFrame: 38407 rows

📄 Sample Orders Data (first 5 rows):


,sr.no,state,district,company,csc_id,distributor_name,distributor_phone_number,vle_name,vle_phone_number,order_number,utr_number,order_date,order_status,estore_name,estore_phone_number,number_of_items_in_the_order,number_of_items_removed,final_invoice_amount,total_offer_amount,discounted_amount,total_amount,payment_method,transaction_id,payment_gateway,gateway_id,transaction_date,transaction_status,is_master_distributor,awd_number,courier_partner,tracking_link,contact_person,contact_number,dispatch_date,aging,grievance_no,refund_anr_number,commis_utr_number,payout_date,amount_before_gst,tcs,tds,cgst,sgst,company_commission_type,commission_value,csc_commission_amount,payable_amount,company_gst_no.,distributor_gst_no.,vle_gst_no.,amount_paid_to_company,alternate_name,alternate_phone_number,lead_bill_date,lead_bill_number,total_remaining_amount
0,1,Bihar,Araria,AWADH ENTERPRISES AND BRAN...,565752450012,Syed Iqrar Husain,8574430100,RAVINDRA KUMAR ROY,8521119574,DI1719334629,AXISP00512383474,2024-06-25,DELIVERED,Awadh Enterprises,8574430100,1,0,1300.0,0.0,0.0,1300.0,ONLINE,9b06bdfbab7cd6cefb7d,ICICI,NaN,2024-06-25,PAID,No,CU550745439IN,India Post,www.indiapost.gov.in,jitendra,9.794213e+09,2024-06-25,0.0,NaN,NaN,NaN,2024-06-28,1101.69,11.02,13.00,5.51,5.51,Percentage,10.0,130.0,1170.0,09ADKPH2514H1ZP,NaN,NaN,NaN,NaN,0.000000e+00,NaN,NaN,NaN
1,2,Bihar,Araria,Go5 Incorporation,365374110017,Go5 Incorporation,8527302352,NIRAJ KUMAR,9262381041,DK1751691291,AXISP00687802353,2025-07-05,CANCELLED,Tecsox,8527302352,1,0,239.0,0.0,0.0,239.0,ONLINE,6d0bee6830ca252e6d47,ICICI,NaN,2025-07-05,PAID,No,NaN,NaN,NaN,Abhishek,9.990266e+09,NaN,NaN,BBGRA1752022334,NaN,NaN,2025-07-07,202.54,1.01,0.23,0.50,0.50,Percentage,10.0,23.9,215.1,06AERPG4388Q1ZK,06AERPG4388Q1ZK,NaN,NaN,Niraj Kumar,7.783080e+09,NaN,NaN,NaN
2,3,Bihar,Araria,Darji Online,757545460019,DARJI ONLINE,9610088970,SAQIB ZIA,9661742850,DB1717229029,AXISP00506066686,2024-06-01,DELIVERED,Darji Online,9610088970,1,0,450.0,0.0,0.0,450.0,ONLINE,12395f4ac68b9183d5ac,PAYU,2.003279e+10,2024-06-01,PAID,No,CR329232605IN,post office,https://www.indiapost.gov.in/,Setha singh rawat,8.829855e+09,2024-06-22,21.0,NaN,NaN,NaN,2024-06-04,0.00,0.00,4.50,0.00,0.00,Percentage,10.0,45.0,405.0,08BZLPR9173C1ZV,08BZLPR9173C1ZV,NaN,NaN,SAQIB ZIA,8.292427e+09,NaN,NaN,NaN
3,4,Bihar,Araria,AUM Imagineering Private L...,517962300011,Jatin Pandya,7600045520,RAVINDRA SAH,8709168847,DZ1740402953,AXISP00623092710,2025-02-24,DELIVERED,Mantra Devices,7600045520,1,0,3270.0,0.0,0.0,3270.0,ONLINE,3ad0f473f53aefdd20c8,ICICI,NaN,2025-02-24,PAID,No,EG279650884IN,INDIA POST,https://www.indiapost.gov....,NaN,NaN,2025-02-25,1.0,NaN,NaN,NaN,2025-02-27,2771.19,13.86,3.27,6.93,6.93,Percentage,0.0,0.0,3270.0,24AAJCA3252C1Z7,NaN,NaN,NaN,Ravindra kumar sah,8.709169e+09,NaN,NaN,NaN
4,5,Bihar,Araria,AUM Imagineering Private L...,353714130011,Jatin Pandya,7600045520,Vipeen Kumar Bhartee,8877877386,DJ1762944560,AXISP00740961013,2025-11-12,DELIVERED,Mantra Devices,7600045520,1,0,3270.0,0.0,0.0,3270.0,ONLINE,5448e57bf7990b8e0ddb,ICICI,NaN,2025-11-12,PAID,No,EG574752920IN,INDIA POST,https://www.indiapost.gov....,NaN,NaN,2025-11-13,1.0,NaN,NaN,NaN,2025-11-13,2771.19,13.86,3.27,6.93,6.93,Percentage,0.0,0.0,3270.0,24AAJCA3252C1Z7,NaN,NaN,NaN,Vipeen Kumar Bhartee,7.004627e+09,NaN,NaN,NaN



📄 Sample VLE ID Data (first 5 rows):


,sr.no,state,district,csc_id,store_name,contact_person,mobile,email,pincode,address_1,address_2,address_3,status
0,1,Bihar,Araria,355265170017,Green Wish Meal Farmer Pro...,Divyanshu Kumar,9199774020,kumardivyanshu3522@gmail.com,854318,NaN,NaN,NaN,ACTIVATED
3,4,Bihar,Araria,134714270012,Taiyab alam,Taiyab Alam,9771221983,taiyabalam167@gmail.com,854331,NaN,NaN,NaN,APPROVED
4,5,Bihar,Araria,655041840017,APNA KENDRA,MD UMAR FAROQUE,7004181633,umarfaroque11@gmail.com,854318,NaN,NaN,NaN,APPROVED
5,6,Bihar,Araria,739131020016,Common Service Center Khor...,Shabbir Alam,9708867368,shbbiralam007@rediffmail.com,854333,NaN,NaN,NaN,COMPLETED
6,7,Bihar,Araria,661184040017,Anchal Digital Seva Gaira,NARESH KUMAR SAH,8292677119,cscgaira@gmail.com,854325,NaN,NaN,NaN,-
